# 03 - Mondrian prototyping

Build and check the digitized rectangle geometry, the decade-assignment
logic, and the final render here, cell by cell, before any of it moves
into `src/`. Nothing in this notebook is wired into the rest of the
project until the graduation step copies the validated code into
`src/mondrian_geometry.py` and `src/charts.py`.

In [1]:
import sys
sys.path.insert(0, "..")

import plotly.graph_objects as go

from src import config

palette = config.PALETTES["mondrian"]

## Geometry

Digitized from `images/mondrian_composition.jpg`. Coordinates are
normalized to [0, 1] in plot space (y0=0 is the bottom of the image,
matching Plotly's axis direction, not image top-left convention).

This is a starting point — adjust coordinates below and re-run the
visual-check cell until it reads as the source image.

In [ ]:
MONDRIAN_RECTANGLES = [
    {"x0": 0.15, "y0": 0.65, "x1": 0.20, "y1": 0.75, "color": "yellow"},
    {"x0": 0.20, "y0": 0.55, "x1": 0.25, "y1": 0.75, "color": "red"},
    {"x0": 0.30, "y0": 0.25, "x1": 0.55, "y1": 0.55, "color": "red"},
    {"x0": 0.55, "y0": 0.55, "x1": 0.65, "y1": 0.75, "color": "yellow"},
    {"x0": 0.75, "y0": 0.65, "x1": 0.80, "y1": 0.75, "color": "blue"},
    {"x0": 0.80, "y0": 0.65, "x1": 0.85, "y1": 0.75, "color": "black"},
    {"x0": 0.65, "y0": 0.35, "x1": 0.80, "y1": 0.55, "color": "red"},
    {"x0": 0.80, "y0": 0.35, "x1": 0.85, "y1": 0.45, "color": "yellow"},
    {"x0": 0.15, "y0": 0.25, "x1": 0.25, "y1": 0.35, "color": "blue"},
    {"x0": 0.25, "y0": 0.25, "x1": 0.30, "y1": 0.55, "color": "yellow"},
    {"x0": 0.55, "y0": 0.25, "x1": 0.60, "y1": 0.35, "color": "black"},
    {"x0": 0.55, "y0": 0.15, "x1": 0.65, "y1": 0.25, "color": "blue"},
    {"x0": 0.25, "y0": 0.15, "x1": 0.30, "y1": 0.25, "color": "yellow"},
]

for rect in MONDRIAN_RECTANGLES:
    assert 0 <= rect["x0"] < rect["x1"] <= 1
    assert 0 <= rect["y0"] < rect["y1"] <= 1
    assert rect["color"] in palette
len(MONDRIAN_RECTANGLES)

## Background fill

Colored-rectangle edges define a fine candidate grid, but white cells are
then **merged**: each free grid cell is greedily expanded as far right,
then as far down, as it stays free, before moving to the next
not-yet-covered cell. A dividing line only survives in the output where a
colored block's real edge forces it — no line cuts through a colored
block's interior, and white space isn't split just because some *other*,
unrelated colored block happens to share that coordinate elsewhere.

In [ ]:
def fill_background_cells(rectangles):
    xs = sorted({r["x0"] for r in rectangles} | {r["x1"] for r in rectangles})
    ys = sorted({r["y0"] for r in rectangles} | {r["y1"] for r in rectangles})
    n_cols, n_rows = len(xs) - 1, len(ys) - 1

    def is_covered(cx, cy):
        return any(r["x0"] <= cx <= r["x1"] and r["y0"] <= cy <= r["y1"] for r in rectangles)

    free = {}
    for i in range(n_cols):
        for j in range(n_rows):
            cx_mid = (xs[i] + xs[i + 1]) / 2
            cy_mid = (ys[j] + ys[j + 1]) / 2
            free[(i, j)] = not is_covered(cx_mid, cy_mid)

    merged = []
    for j in range(n_rows):
        i = 0
        while i < n_cols:
            if not free[(i, j)]:
                i += 1
                continue

            i_end = i
            while i_end + 1 < n_cols and free[(i_end + 1, j)]:
                i_end += 1

            j_end = j
            while j_end + 1 < n_rows and all(free[(k, j_end + 1)] for k in range(i, i_end + 1)):
                j_end += 1

            for jj in range(j, j_end + 1):
                for ii in range(i, i_end + 1):
                    free[(ii, jj)] = False

            merged.append({
                "x0": xs[i], "y0": ys[j],
                "x1": xs[i_end + 1], "y1": ys[j_end + 1],
                "color": "background",
            })
            i = i_end + 1
    return merged


MONDRIAN_BACKGROUND_RECTANGLES = fill_background_cells(MONDRIAN_RECTANGLES)
len(MONDRIAN_BACKGROUND_RECTANGLES)

## Line overshoots

Decorative only — echoes the reference image's black lines that poke past
the grid. Every internal grid line that reaches the top/bottom/left/right
edge of the whole composition gets a short stroke continuing past that
edge, with a random extra length.

Two perpendicular ticks are then added, deterministically (no
randomness): each picks the **longest** line of its orientation (one
vertical, one horizontal) as the base, and the **longest line shorter
than that base** — any orientation — as the reference, skipping any
reference so close in length to the base that almost nothing would be
left beyond the crossing point (that reads as a "T", not a "+", once
drawn at the line's thickness). The tick crosses the base at a distance
from its edge equal to the reference's length, and the tick's own length
is the reference's length too.

In [ ]:
import random as _random

GRID_LINE_WIDTH = 5
MIN_ARM_GAP = 0.03  # minimum remaining length beyond the crossing point


def line_overshoots(colored, seed=config.RANDOM_STATE, min_extra=0.08, max_extra=0.20):
    """Each overshoot is a dict, not a raw point pair: {orientation,
    fixed, edge, tip}. "vertical" means x is fixed and edge/tip are y
    values (a top/bottom overshoot); "horizontal" means y is fixed and
    edge/tip are x values (a left/right overshoot). Storing it this way
    means downstream code never has to re-derive which side a line is on
    or which of its two endpoints is the edge vs. the tip."""
    rng = _random.Random(seed)
    xs = sorted({r["x0"] for r in colored} | {r["x1"] for r in colored})
    ys = sorted({r["y0"] for r in colored} | {r["y1"] for r in colored})
    x_min, x_max = xs[0], xs[-1]
    y_min, y_max = ys[0], ys[-1]

    lines = []
    for x in xs[1:-1]:
        if any(r["y1"] == y_max and x in (r["x0"], r["x1"]) for r in colored):
            lines.append({"orientation": "vertical", "fixed": x,
                           "edge": y_max, "tip": y_max + rng.uniform(min_extra, max_extra)})
        if any(r["y0"] == y_min and x in (r["x0"], r["x1"]) for r in colored):
            lines.append({"orientation": "vertical", "fixed": x,
                           "edge": y_min, "tip": y_min - rng.uniform(min_extra, max_extra)})
    for y in ys[1:-1]:
        if any(r["x0"] == x_min and y in (r["y0"], r["y1"]) for r in colored):
            lines.append({"orientation": "horizontal", "fixed": y,
                           "edge": x_min, "tip": x_min - rng.uniform(min_extra, max_extra)})
        if any(r["x1"] == x_max and y in (r["y0"], r["y1"]) for r in colored):
            lines.append({"orientation": "horizontal", "fixed": y,
                           "edge": x_max, "tip": x_max + rng.uniform(min_extra, max_extra)})
    return lines


def line_length(line):
    return abs(line["tip"] - line["edge"])


def line_segment(line):
    a, b = line["edge"], line["tip"]
    if line["orientation"] == "vertical":
        return (line["fixed"], a), (line["fixed"], b)
    return (a, line["fixed"]), (b, line["fixed"])


def free_cross_ticks(lines):
    """Deterministic: each tick's base is the longest line of its own
    orientation (one vertical, one horizontal); its reference is the
    longest line -- any orientation -- that's shorter than the base by
    at least MIN_ARM_GAP (so the arm beyond the crossing point is always
    long enough to read as a "+", not a "T"). The tick crosses the base
    at a distance from its edge equal to the reference's length, and the
    tick's own length is the reference's length too."""

    def make_tick(orientation):
        same_orientation = [l for l in lines if l["orientation"] == orientation]
        if not same_orientation:
            return None
        base = max(same_orientation, key=line_length)
        base_len = line_length(base)
        candidates = [
            l for l in lines
            if l is not base and line_length(l) <= base_len - MIN_ARM_GAP
        ]
        if not candidates:
            return None
        reference = max(candidates, key=line_length)
        ref_len = line_length(reference)
        direction = 1 if base["tip"] >= base["edge"] else -1
        cross_at = base["edge"] + ref_len * direction
        return {
            "orientation": "horizontal" if orientation == "vertical" else "vertical",
            "fixed": cross_at,
            "edge": base["fixed"] - ref_len / 2,
            "tip": base["fixed"] + ref_len / 2,
        }

    return [t for t in (make_tick("vertical"), make_tick("horizontal")) if t is not None]


def segments_trace(segments, color, width=GRID_LINE_WIDTH):
    xs, ys = [], []
    for (sx0, sy0), (sx1, sy1) in segments:
        xs += [sx0, sx1, None]
        ys += [sy0, sy1, None]
    return go.Scatter(x=xs, y=ys, mode="lines", line=dict(color=color, width=width), hoverinfo="skip", showlegend=False)


MONDRIAN_LINE_OVERSHOOTS = line_overshoots(MONDRIAN_RECTANGLES)
MONDRIAN_CROSS_TICKS = free_cross_ticks(MONDRIAN_LINE_OVERSHOOTS)
len(MONDRIAN_LINE_OVERSHOOTS), len(MONDRIAN_CROSS_TICKS)

## Open corners

Background rectangles are still filled solid, but their border is no
longer one closed loop. Each of the 4 corners is checked: if a colored
block's boundary passes through that exact point, the corner stays
joined (it's needed to line up against the color block). If a corner is
touched only by other background rectangles, the two edges meeting there
are pulled back slightly instead of joining — a small deliberate gap.

In [ ]:
def _on_colored_boundary(x, y, colored):
    for r in colored:
        on_vertical_edge = x in (r["x0"], r["x1"]) and r["y0"] <= y <= r["y1"]
        on_horizontal_edge = y in (r["y0"], r["y1"]) and r["x0"] <= x <= r["x1"]
        if on_vertical_edge or on_horizontal_edge:
            return True
    return False


def _shrink_segment(start, end, gap_at_start, gap_at_end):
    (x0, y0), (x1, y1) = start, end
    dx, dy = x1 - x0, y1 - y0
    length = (dx ** 2 + dy ** 2) ** 0.5
    if length == 0:
        return start, end
    ux, uy = dx / length, dy / length
    new_start = (x0 + ux * gap_at_start, y0 + uy * gap_at_start)
    new_end = (x1 - ux * gap_at_end, y1 - uy * gap_at_end)
    return new_start, new_end


def background_edge_segments(rect, colored, gap_fraction=1 / 2):
    x0, y0, x1, y1 = rect["x0"], rect["y0"], rect["x1"], rect["y1"]
    corners = {"bl": (x0, y0), "br": (x1, y0), "tr": (x1, y1), "tl": (x0, y1)}
    open_corner = {
        name: not _on_colored_boundary(x, y, colored) for name, (x, y) in corners.items()
    }

    edges = [
        (corners["bl"], corners["br"], "bl", "br"),
        (corners["br"], corners["tr"], "br", "tr"),
        (corners["tr"], corners["tl"], "tr", "tl"),
        (corners["tl"], corners["bl"], "tl", "bl"),
    ]
    segments = []
    for start, end, start_name, end_name in edges:
        length = ((end[0] - start[0]) ** 2 + (end[1] - start[1]) ** 2) ** 0.5
        gap_start = length * gap_fraction if open_corner[start_name] else 0
        gap_end = length * gap_fraction if open_corner[end_name] else 0
        segments.append(_shrink_segment(start, end, gap_start, gap_end))
    return segments


def fill_only_trace(rect, palette):
    x0, y0, x1, y1 = rect["x0"], rect["y0"], rect["x1"], rect["y1"]
    return go.Scatter(
        x=[x0, x1, x1, x0, x0],
        y=[y0, y0, y1, y1, y0],
        fill="toself",
        fillcolor=palette[rect["color"]],
        line=dict(width=0),
        mode="lines",
        hoverinfo="skip",
        showlegend=False,
    )

## Visual check

Compare against `images/mondrian_composition.jpg`. If a rectangle looks
off, adjust its coordinates in the cell above and re-run this cell.

In [ ]:
def rectangle_trace(rect, hovertemplate="<extra></extra>", text=None):
    x0, y0, x1, y1 = rect["x0"], rect["y0"], rect["x1"], rect["y1"]
    points = [x0, x1, x1, x0, x0]
    return go.Scatter(
        x=points,
        y=[y0, y0, y1, y1, y0],
        fill="toself",
        fillcolor=palette[rect["color"]],
        line=dict(color=palette["black"], width=GRID_LINE_WIDTH),
        mode="lines",
        hovertemplate=hovertemplate,
        text=[text] * len(points) if text is not None else None,
        showlegend=False,
    )

x_min = min(r["x0"] for r in MONDRIAN_RECTANGLES)
x_max = max(r["x1"] for r in MONDRIAN_RECTANGLES)
y_min = min(r["y0"] for r in MONDRIAN_RECTANGLES)
y_max = max(r["y1"] for r in MONDRIAN_RECTANGLES)

decorative_segments = [line_segment(l) for l in MONDRIAN_LINE_OVERSHOOTS + MONDRIAN_CROSS_TICKS]
overshoot_xs = [p[0] for seg in decorative_segments for p in seg]
overshoot_ys = [p[1] for seg in decorative_segments for p in seg]
plot_x_min = min([x_min] + overshoot_xs)
plot_x_max = max([x_max] + overshoot_xs)
plot_y_min = min([y_min] + overshoot_ys)
plot_y_max = max([y_max] + overshoot_ys)

fig = go.Figure()
for rect in MONDRIAN_BACKGROUND_RECTANGLES:
    fig.add_trace(fill_only_trace(rect, palette))
    fig.add_trace(segments_trace(background_edge_segments(rect, MONDRIAN_RECTANGLES), palette["black"]))
for rect in MONDRIAN_RECTANGLES:
    fig.add_trace(rectangle_trace(rect))
fig.add_trace(segments_trace(decorative_segments, palette["black"]))
fig.update_layout(
    plot_bgcolor=palette["background"],
    xaxis=dict(visible=False, range=[plot_x_min, plot_x_max]),
    yaxis=dict(visible=False, range=[plot_y_min, plot_y_max], scaleanchor="x"),
    showlegend=False,
    margin=dict(t=20, l=0, r=0, b=0),
)
fig.show()

## Decade-to-rectangle assignment

Geometry stays fixed — assignment is rank-based, not size-based. The
largest rectangle gets the most-acquired decade, ranked on down. If there
are more decades than rectangles, the smallest decades pool into a single
"Other" entry first (then the result is re-sorted, since the pooled total
can outrank an individually-kept decade). If there are more rectangles
than decades, the smallest-ranked rectangles are left unassigned.

In [ ]:
def _assign_decades_to_rectangles(decade_counts, n_rectangles):
    items = sorted(decade_counts.items(), key=lambda kv: kv[1], reverse=True)
    if len(items) > n_rectangles:
        keep = items[: n_rectangles - 1]
        other_total = sum(count for _, count in items[n_rectangles - 1:])
        items = keep + [("Other", other_total)]
        items.sort(key=lambda kv: kv[1], reverse=True)
    assignments = list(items) + [None] * (n_rectangles - len(items))
    return assignments[:n_rectangles]

In [ ]:
# Exact match: ranks straightforwardly by count
assert _assign_decades_to_rectangles({"1990s": 50, "1960s": 100, "2000s": 20}, 3) == \
    [("1960s", 100), ("1990s", 50), ("2000s", 20)]

# More decades than rectangles: smallest pool into "Other"
assert _assign_decades_to_rectangles(
    {"1960s": 100, "1970s": 80, "1980s": 10, "1990s": 5, "2000s": 3}, 3
) == [("1960s", 100), ("1970s", 80), ("Other", 18)]

# "Other"'s pooled total can outrank individually-kept decades -- must re-sort
assert _assign_decades_to_rectangles(
    {"1960s": 50, "1970s": 40, "1980s": 30, "1990s": 25, "2000s": 20}, 3
) == [("Other", 75), ("1960s", 50), ("1970s", 40)]

# More rectangles than decades: leftover rectangles get no data
assert _assign_decades_to_rectangles({"1960s": 100, "1970s": 80}, 4) == \
    [("1960s", 100), ("1970s", 80), None, None]

print("all assignment checks passed")

## Final render, with real data

In [ ]:
from src import data

df = data.load_raw_data()
cleaned = data.clean_artworks(df)
cleaned["Decade_acquired"].value_counts()

In [ ]:
def mondrian_treemap(df):
    known = df[df["Decade_acquired"] != "unknown"]
    decade_counts = known["Decade_acquired"].value_counts().to_dict()

    rectangles_sorted = sorted(
        MONDRIAN_RECTANGLES,
        key=lambda r: (r["x1"] - r["x0"]) * (r["y1"] - r["y0"]),
        reverse=True,
    )
    assignments = _assign_decades_to_rectangles(decade_counts, len(rectangles_sorted))

    x_min = min(r["x0"] for r in MONDRIAN_RECTANGLES)
    x_max = max(r["x1"] for r in MONDRIAN_RECTANGLES)
    y_min = min(r["y0"] for r in MONDRIAN_RECTANGLES)
    y_max = max(r["y1"] for r in MONDRIAN_RECTANGLES)
    decorative_segments = [line_segment(l) for l in MONDRIAN_LINE_OVERSHOOTS + MONDRIAN_CROSS_TICKS]
    overshoot_xs = [p[0] for seg in decorative_segments for p in seg]
    overshoot_ys = [p[1] for seg in decorative_segments for p in seg]
    plot_x_min = min([x_min] + overshoot_xs)
    plot_x_max = max([x_max] + overshoot_xs)
    plot_y_min = min([y_min] + overshoot_ys)
    plot_y_max = max([y_max] + overshoot_ys)

    fig = go.Figure()
    for rect in MONDRIAN_BACKGROUND_RECTANGLES:
        fig.add_trace(fill_only_trace(rect, palette))
        fig.add_trace(segments_trace(background_edge_segments(rect, MONDRIAN_RECTANGLES), palette["black"]))
    for rect, assignment in zip(rectangles_sorted, assignments):
        if assignment is not None:
            decade, count = assignment
            hovertemplate = f"{decade}<br>%{{text}} artworks<extra></extra>"
            text = str(count)
        else:
            hovertemplate = "<extra></extra>"
            text = None
        fig.add_trace(rectangle_trace(rect, hovertemplate=hovertemplate, text=text))
    fig.add_trace(segments_trace(decorative_segments, palette["black"]))

    fig.update_layout(
        plot_bgcolor=palette["background"],
        xaxis=dict(visible=False, range=[plot_x_min, plot_x_max]),
        yaxis=dict(visible=False, range=[plot_y_min, plot_y_max], scaleanchor="x"),
        showlegend=False,
        margin=dict(t=20, l=0, r=0, b=0),
    )
    return fig

mondrian_treemap(cleaned).show()

In [ ]:
import plotly

print("plotly version:", plotly.__version__)

debug_fig = mondrian_treemap(cleaned)
print("total traces:", len(debug_fig.data))
for i, trace in enumerate(debug_fig.data):
    ht = getattr(trace, "hovertemplate", None)
    tx = getattr(trace, "text", None)
    if ht not in (None, "<extra></extra>"):
        print(i, type(trace).__name__, "hovertemplate=", repr(ht), "text=", repr(tx))

In [ ]:
import webbrowser

html_path = "mondrian_treemap_preview.html"
mondrian_treemap(cleaned).write_html(html_path, include_plotlyjs="cdn")
webbrowser.open(html_path)
print("opened:", html_path)